# PAD test run

Prepares a fresh Colab runtime, rehearses the whole pipeline on the **validation** patients, and only then
runs the single held-out **test** evaluation.

**Before cell 1:** Runtime -> Change runtime type -> **GPU**. Doing it later wipes `/content` and every cell
here has to run again.

Run cells 1-5 in order, then cell 6. **Cell 8 is the test run: it happens once and cannot be undone.**
Do not run it unless cell 6 printed `REHEARSAL PASSED`.

## 1 · The runtime

In [1]:
import torch
print(torch.__version__,
      "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU",
      "| bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU, then start again from here."
assert "+cpu" not in torch.__version__, "CPU-only torch; see cell 2 - do not install requirements.txt here."

2.11.0+cu130 | cuda: True | Tesla T4 | bf16: True


## 2 · Packages

Only the three the repo needs on top of Colab's own stack. **Not** `requirements.txt`: that file pins
`torch==2.11.0` for the local training machine and would replace Colab's CUDA build with a CPU one.

If the printout does not say `sklearn 1.9.1`, use Runtime -> Restart session and run this cell again.
`models/questionnaire.joblib` was written with 1.9.1 and will not unpickle cleanly under a different version.

In [2]:
!pip install -q scikit-learn==1.9.1 timm==1.0.30 safetensors

import sklearn, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| sklearn", sklearn.__version__)
assert sklearn.__version__ == "1.9.1", "restart the session and run this cell again"

torch 2.11.0+cu130 | cuda True | sklearn 1.9.1


## 3 · The repository

The clone fails harmlessly when the directory already exists, as it does after a restart; the four size
checks below it are what decide whether this runtime has the right repository.

In [3]:
!git clone https://github.com/IfatDav/skin-lesion-assistant.git /content/repo

from pathlib import Path
for f, size in [("src/rehearse_val.py", 4460), ("src/evaluate_test.py", 4450),
                ("data/splits/pad_split.csv", 140195), ("data/splits/image_index.csv", None)]:
    p = Path("/content/repo") / f
    got = p.stat().st_size if p.exists() else 0
    print(f"  {'OK ' if got else '** MISSING **'} {f:<30} {got:>8} bytes"
          + (f"  (expected {size})" if size else ""))

Cloning into '/content/repo'...
remote: Enumerating objects: 937, done.
remote: Counting objects: 100% (107/107), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 937 (delta 50), reused 6 (delta 6), pack-reused 830 (from 2)
Receiving objects: 100% (937/937), 19.33 MiB | 29.58 MiB/s, done.
Resolving deltas: 100% (383/383), done.
  OK  src/rehearse_val.py                4459 bytes  (expected 4460)
  OK  src/evaluate_test.py               4450 bytes  (expected 4450)
  OK  data/splits/pad_split.csv        140195 bytes  (expected 140195)
  OK  data/splits/image_index.csv     1274761 bytes


## 4 · The PAD images and metadata

Both come from the public mirror, onto local disk. Not from Drive: the images are read one file at a time
and Drive's FUSE layer is bound by file count, not bandwidth.

The mirror's `metadata.csv` is the official PAD-UFES-20 table, and `src/pad_split.py` cross-checks it
against the team's frozen split on twelve counts before the model is even loaded. A mismatch stops the run
there, so nothing downstream can quietly use the wrong metadata.

In [4]:
import os, glob, shutil
from pathlib import Path
import pandas as pd
from huggingface_hub import snapshot_download

DATA   = Path("/content/DATA")
TARGET = DATA / "metadata" / "PAD_UFES_20_metadata.csv"
PADDIR = DATA / "pad_original" / "pad_original"
TARGET.parent.mkdir(parents=True, exist_ok=True)
PADDIR.mkdir(parents=True, exist_ok=True)

mirror = Path(snapshot_download(repo_id="SalmaneExploring/pad-ufes-20", repo_type="dataset",
                                allow_patterns=["all_images/**", "metadata.csv"],
                                local_dir="/content/pad_mirror"))
shutil.copy(mirror / "metadata.csv", TARGET)

for src in glob.glob(f"{mirror}/all_images/**/*.png", recursive=True):
    dst = PADDIR / Path(src).name
    if not dst.exists():
        try:
            os.link(src, dst)          # instant, no extra disk
        except OSError:
            shutil.copy(src, dst)

md_ = pd.read_csv(TARGET)
n_img = len(list(PADDIR.glob("*.png")))
print(f"metadata: {len(md_)} rows, {len(md_.columns)} columns  (expected 2298, 26)")
print(f"images  : {n_img}  (expected 2298)")
assert len(md_) == 2298 and n_img == 2298, "staging incomplete - run this cell again"

os.environ["SKIN_DATA_ROOT"] = str(DATA)
print("SKIN_DATA_ROOT =", os.environ["SKIN_DATA_ROOT"])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2299 files:   0%|          | 0/2299 [00:00<?, ?it/s]

metadata: 2298 rows, 26 columns  (expected 2298, 26)
images  : 2298  (expected 2298)
SKIN_DATA_ROOT = /content/DATA


## 5 · Precision

`src/model.py` asks for bfloat16, which Turing cards (the free T4) do not support. On those this swaps in
float16; on an L4 or A100 it changes nothing and the arithmetic stays identical to the validation run that
produced `reports/final_results.json`. Local to this runtime - the repository is untouched.

In [5]:
import torch
from pathlib import Path

# 1. bf16 -> fp16 on pre-Ampere cards. is_bf16_supported() returns True on a T4 because of
#    emulation, but cuDNN has no bf16 convolution kernel before Ampere.
major, minor = torch.cuda.get_device_capability()
m = Path("/content/repo/src/model.py")
if major < 8:
    m.write_text(m.read_text().replace("dtype=torch.bfloat16", "dtype=torch.float16"))
    print(f"compute capability {major}.{minor} (pre-Ampere) -> patched src/model.py to float16")
else:
    print(f"compute capability {major}.{minor} -> bf16 kernels available, no patch needed")

# 2. this runtime has 2 vCPUs; the default of 8 dataloader workers risks a freeze
d = Path("/content/repo/src/data.py")
d.write_text(d.read_text().replace("workers: int = 8", "workers: int = 2"))
print("dataloader workers 8 -> 2")

compute capability 7.5 (pre-Ampere) -> patched src/model.py to float16
dataloader workers 8 -> 2


In [11]:
%%writefile /content/repo/src/dump_val_scores.py
"""Per-patient validation scores, for the M1 queue simulation.

Writes what the evaluation already computes; it fits nothing and touches no test patient.
    python -m src.dump_val_scores
"""
import json

import numpy as np
import pandas as pd

from src import config as C
from src.data import load_index, make_loader
from src.fusion import fuse
from src.metrics import cancer_score
from src.model import predict
from src.operating_point import to_patients
from src.predict import load_final_models
from src.questionnaire import patient_scores

OUT = C.REPO_ROOT / "data" / "processed" / "val_scores_fused.csv"


def main():
    w = json.loads((C.MODELS_DIR / "final_model.json").read_text())["fusion"]["w_image"]
    val = load_index("pad_val")
    p = np.mean([predict(m.cuda(), make_loader(val, train=False)).numpy()
                 for m in load_final_models()], axis=0)
    img = to_patients(val["patient_id"].to_numpy(),
                      (val["cancer"] == "1").astype(int).to_numpy(), cancer_score(p))
    quest = patient_scores("val")
    common = img.index.intersection(quest.index)

    df = pd.DataFrame({"y": img["y"].loc[common],
                       "image_score": img["score"].loc[common],
                       "questionnaire_score": quest["score"].loc[common]})
    df["fused_rank_in_set"] = fuse(df["image_score"], df["questionnaire_score"], w)
    df.index.name = "patient_id"
    df.to_csv(OUT)

    print(f"wrote {OUT.relative_to(C.REPO_ROOT)}")
    print(f"{len(df)} patients, {int(df['y'].sum())} malignant  (expected 220, 103)")
    print(df.head())


if __name__ == "__main__":
    main()

Writing /content/repo/src/dump_val_scores.py


In [12]:
!SKIN_DATA_ROOT=/content/DATA python -m src.dump_val_scores

wrote data/processed/val_scores_fused.csv
220 patients, 103 malignant  (expected 220, 103)
            y  image_score  questionnaire_score  fused_rank_in_set
patient_id                                                        
PAT_100     0     0.397509             0.049724           0.247273
PAT_1014    0     0.104939             0.040149           0.100000
PAT_1017    0     0.372179             0.205630           0.325455
PAT_1031    0     0.152707             0.034816           0.125455
PAT_1046    0     0.189555             0.142337           0.217273


## 6 · Dress rehearsal — validation patients only

The same code path as the test run, on the 220 validation patients, writing nothing. It downloads the
111 MB checkpoint from the GitHub release on first use and checks its sha256, so this cell takes a few
minutes. It must reproduce the published numbers: image 0.9556, questionnaire 0.8668, fused 0.9542.

In [6]:
%cd /content/repo
!SKIN_DATA_ROOT=/content/DATA python -m src.rehearse_val

/content/repo
torch 2.11.0+cu130 | cuda True | Tesla T4 | bf16 True
scikit-learn 1.9.1 (models/questionnaire.joblib was written with 1.9.1)
frozen PAD manifest at data/splits/pad_split.csv: True

validation images: 355 (expected 355)
downloading convnext_pad_s42_scc2.safetensors from https://github.com/IfatDav/skin-lesion-assistant/releases/download/image-model-v1/
patients: 220 (expected 220) | malignant: 103 (expected 103)

  OK   image          AUC 0.9536 vs 0.9556 (-0.0020) | sens@10 slots 0.592 vs 0.602 (-0.010)
  OK   questionnaire  AUC 0.8668 vs 0.8668 (+0.0000) | sens@10 slots 0.524 vs 0.524 (+0.000)
  OK   final_fused    AUC 0.9537 vs 0.9542 (-0.0005) | sens@10 slots 0.592 vs 0.612 (-0.020)
       image - questionnaire  +0.0867 vs +0.0888 | CI [+0.045, +0.130]

REHEARSAL PASSED. This environment reproduces the published validation numbers;
`python -m src.evaluate_test --frozen` can run.


## 7 · Stop and read the result

| Cell 6 printed | What to do |
|---|---|
| `REHEARSAL PASSED` | Go on to cell 8. |
| any `DIFF` line | **Stop.** This environment does not reproduce the published numbers. Send the output to Claude before touching the test set. |
| a crash | **Stop.** Nothing was lost; fix the cause and run cell 6 again. |

The test set is used once. `src/evaluate_test.py` refuses to run a second time, and there is no way back
from a run made on a broken setup.

## 8 · The single test run

`reports/decisions.md` was approved on 26.10.2026 and the decisions are frozen. This writes
`reports/test_results.json` and the blind ink-labelling sheet, and will refuse to run if the results file
already exists.

In [7]:
%cd /content/repo
!SKIN_DATA_ROOT=/content/DATA python -m src.evaluate_test --frozen

/content/repo
reports/test_results.json already exists: the test set is used once.


## 9 · Check and save

**The first thing to check is 275 patients and 128 with cancer.** Any other number means the patient
intersection was incomplete and the result does not stand.

> **`expected 129` in the cell below is a stale string, written before the run. The correct number is 128,
> and the run is valid.**
>
> PAD stores one photograph under two lesion ids with contradictory diagnoses:
> `PAT_759_1538_566.png` (BCC, nose, biopsied) is byte-identical to `PAT_759_1433_914.png` (NEV, back,
> biopsied) — same md5, same 247,030 bytes. The de-duplication in `src/build_index.py` keeps the benign
> row, so PAT_759 is scored as a negative. The frozen split manifest counts 129 malignant test patients;
> the de-duplicated index evaluates **128**. Twelve of PAD's fifteen byte-identical pairs carry
> contradictory diagnoses, but this is the only one that crosses the cancer boundary.
>
> The error penalises the model — a correct high score on that photograph counts as a false alarm — so it
> cannot inflate the result, and the index was left untouched. Recorded in `reports/decisions.md` **before**
> this run.

In [8]:
import json
from pathlib import Path

r = json.loads(Path("/content/repo/reports/test_results.json").read_text())

a = r["all_patients"]
print(f"ALL PATIENTS   {a['n_patients']} patients (expected 275), "
      f"{a['n_malignant']} with cancer (expected 129)")
for k in ("final_fused", "image", "questionnaire"):
    s = a[k]
    print(f"  {k:<14} AUC {s['auc']:.3f} ({s['auc_ci'][0]:.3f}-{s['auc_ci'][1]:.3f})"
          f" | sens@10 slots {s['primary']['sensitivity']:.2f} | PPV {s['primary']['ppv']:.2f}")

b = r["biopsied_only"]
print(f"\nBIOPSIED ONLY  {b['n_patients']} patients, {b['n_malignant']} with cancer")
for k in ("final_fused", "image", "questionnaire"):
    print(f"  {k:<14} AUC {b[k]['auc']:.3f}")

for k in ("final_fused_minus_image", "final_fused_minus_questionnaire", "image_minus_questionnaire"):
    d = a[k]
    print(f"\n{k:<34} {d['mean']:+.4f}  CI [{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}]")

ALL PATIENTS   275 patients (expected 275), 128 with cancer (expected 129)
  final_fused    AUC 0.954 (0.928-0.975) | sens@10 slots 0.66 | PPV 0.68
  image          AUC 0.953 (0.929-0.972) | sens@10 slots 0.69 | PPV 0.69
  questionnaire  AUC 0.862 (0.813-0.902) | sens@10 slots 0.57 | PPV 0.57

BIOPSIED ONLY  138 patients, 115 with cancer
  final_fused    AUC 0.851
  image          AUC 0.859
  questionnaire  AUC 0.767

final_fused_minus_image            +0.0004  CI [-0.014, +0.015]

final_fused_minus_questionnaire    +0.0924  CI [+0.063, +0.125]

image_minus_questionnaire          +0.0920  CI [+0.052, +0.136]


### The biopsied-only subgroup, and the capacity grid

The paired differences printed above are for all 275 patients. These are for the hard cases, plus the
slot x prevalence grid the charter's M2 is stated against.

In [9]:
import json
from pathlib import Path

r = json.loads(Path("/content/repo/reports/test_results.json").read_text())

b = r["biopsied_only"]
print(f"BIOPSIED ONLY  {b['n_patients']} patients, {b['n_malignant']} with cancer")
for k in ("final_fused", "image", "questionnaire"):
    s = b[k]
    print(f"  {k:<14} AUC {s['auc']:.3f} ({s['auc_ci'][0]:.3f}-{s['auc_ci'][1]:.3f})"
          f" | sens@10 {s['primary']['sensitivity']:.2f} | PPV {s['primary']['ppv']:.2f}")
for k in ("image_minus_questionnaire", "final_fused_minus_questionnaire", "final_fused_minus_image"):
    d = b[k]
    print(f"  {k:<34} {d['mean']:+.4f}  CI [{d['ci'][0]:+.3f}, {d['ci'][1]:+.3f}]")

print("\nSCENARIOS - final model vs questionnaire, sensitivity at 10% prevalence")
print("  slots  ceiling   fused   questionnaire")
for slots in (5, 10, 15):
    f = next(s for s in r["all_patients"]["final_fused"]["scenarios"]
             if s["slots"] == slots and s["prevalence"] == 0.1)
    q = next(s for s in r["all_patients"]["questionnaire"]["scenarios"]
             if s["slots"] == slots and s["prevalence"] == 0.1)
    print(f"  {slots:<6} {f['ceiling']:<9.2f} {f['sensitivity']:<7.2f} {q['sensitivity']:.2f}")
f20 = next(s for s in r["all_patients"]["final_fused"]["scenarios"]
           if s["slots"] == 10 and s["prevalence"] == 0.2)
print(f"  10 slots at 20% prevalence: ceiling {f20['ceiling']:.2f}, delivered {f20['sensitivity']:.2f}")

BIOPSIED ONLY  138 patients, 115 with cancer
  final_fused    AUC 0.851 (0.742-0.940) | sens@10 0.40 | PPV 0.51
  image          AUC 0.859 (0.769-0.934) | sens@10 0.46 | PPV 0.54
  questionnaire  AUC 0.767 (0.653-0.880) | sens@10 0.22 | PPV 0.22
  image_minus_questionnaire          +0.0931  CI [+0.005, +0.178]
  final_fused_minus_questionnaire    +0.0847  CI [+0.027, +0.143]
  final_fused_minus_image            -0.0083  CI [-0.045, +0.026]

SCENARIOS - final model vs questionnaire, sensitivity at 10% prevalence
  slots  ceiling   fused   questionnaire
  5      0.50      0.39    0.37
  10     1.00      0.66    0.57
  15     1.00      0.82    0.65
  10 slots at 20% prevalence: ceiling 0.50, delivered 0.45


### Get the two files off the runtime before it dies

They are the only record of a run that cannot be repeated. Download them, then commit both to the
repository.

In [10]:
from google.colab import files
files.download("/content/repo/reports/test_results.json")
files.download("/content/repo/evaluation/ink_labeling_test.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
from google.colab import files
files.download("/content/repo/data/processed/val_scores_fused.csv")
files.download("/content/repo/src/dump_val_scores.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10 · What the run said

**275 held-out PAD patients, 128 with a malignant lesion.** Decisions frozen in `reports/decisions.md`
and approved 26.10.2026; run 30.10.2026; run once.

| Model | Patient AUC (95% CI) | Sensitivity @ 10 slots | PPV | Validation |
|---|---|---|---|---|
| Questionnaire only — the baseline | 0.862 (0.813–0.902) | 0.57 | 0.57 | 0.867 |
| Image model | 0.953 (0.929–0.972) | 0.69 | 0.69 | 0.956 |
| **Final: image 60% + questionnaire 40%** | **0.954** (0.928–0.975) | **0.66** | **0.68** | 0.954 |

| Paired, same patients | ΔAUC | 95% CI | Predicted from validation |
|---|---|---|---|
| image − questionnaire | **+0.092** | +0.052 to +0.136 | +0.089 |
| fused − questionnaire | +0.092 | +0.063 to +0.125 | +0.087 |
| fused − image | +0.000 | −0.014 to +0.015 | −0.001 |

**Nothing moved.** The epochs, the SCC class weight and the seed were all chosen on the validation
patients, and the held-out set reproduces every headline number and every paired difference — including
the one the team predicted in advance would be zero. The 60/40 fusion ties the image model on test exactly
as it did on validation, so the reason recorded for choosing it still holds: robustness when a photograph
is poor, not performance.

### On hard cases

PAD's unbiopsied lesions are all benign and easy. On the 138 test patients whose lesions were **all**
biopsied (115 of them with cancer):

| | AUC (95% CI) | Sensitivity @ 10 slots |
|---|---|---|
| Image model | 0.859 (0.769–0.934) | **0.46** |
| Final model | 0.851 (0.742–0.940) | 0.40 |
| Questionnaire | 0.767 (0.653–0.880) | **0.22** |

`image − questionnaire` is **+0.093, CI +0.005 to +0.178**. The interval clears zero by 0.005, on 23
benign patients: a real result and a fragile one, to be quoted with its lower bound rather than as
"significant". On validation this gap had shrunk to four points and lost significance, and the team wrote
that down before opening the test set. On held-out data it did not shrink.

The operating point states it more plainly than the AUC does. On the patients a dermatologist could not
settle by eye, the questionnaire puts **22%** of cancer patients on the daily list and the image model
puts **46%**.

### M2 and the capacity ceiling

| Slots per 100 | Prevalence | Ceiling | Delivered |
|---|---|---|---|
| 5 | 10% | 0.50 | 0.39 |
| **10** | **10%** | **1.00** | **0.66** |
| 15 | 10% | 1.00 | 0.82 |
| 10 | 20% | 0.50 | 0.45 |

The charter's target of ≥ 0.90 is not reached at 10 slots. The constraint is capacity rather than ranking
— no ranking can exceed `budget ÷ prevalence` — so the number of urgent slots required is reported as a
product requirement rather than recorded as a model failure.

### Caveats that travel with these numbers

- **float16, not bfloat16.** Executed on a Tesla T4, which has no cuDNN bfloat16 convolution kernel;
  `torch.cuda.is_bf16_supported()` reports `True` there because of emulation, which is why cell 5 tests
  the compute capability instead. The rehearsal in cell 6 put the cost of the substitution at **−0.002**
  patient AUC, inside the bootstrap interval.
- **Two local patches**, applied in cell 5 and not committed to the repository: the precision swap above,
  and the dataloader worker count lowered from 8 to 2 for a 2-vCPU runtime.
- **One patient is mislabelled by the archive** — PAT_759, see the note above cell 9.
- **Ink is still unmeasured.** `evaluation/ink_labeling_test.csv` was written blind by this run and is the
  input to the shortcut check that follows.

### What happens next

1. ~~Commit `reports/test_results.json` and `evaluation/ink_labeling_test.csv`.~~ **Done 30.10.2026.**
2. ~~Carry the test figures into `reports/results.md`.~~ **Done 30.10.2026.**
3. ~~Carry them into `README.md` and `reports/results_image_model.md`.~~
4. Label the ink sheet (`evaluation/INK_LABELING.md`), then run `python -m src.ink_effect`.